# Attention Vector Extraction — Colab

Extracts Q, K, V attention vectors from **Llama 3.1 8B**.

**How it works:**
- Extracts to Colab's **local SSD** (no Google Drive needed)
- After each task finishes: **zips it in background** while the next task starts extracting
- All zips land in `/content/zips/` — download them at the end or after each phase
- If the runtime dies, re-run: already-downloaded tasks are safe on your machine

**Requirements:**
- **A100 GPU** (40GB+ VRAM). T4 cannot handle 128K context.
- ~50 GB local disk (Colab provides ~100-350 GB)

## 1. Check GPU

In [ ]:
import torch, shutil, os

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Go to Runtime > Change runtime type > A100."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
disk_free_gb = shutil.disk_usage("/content").free / 1e9

print(f"GPU:  {gpu_name} ({gpu_mem_gb:.0f} GB VRAM)")
print(f"Disk: {disk_free_gb:.0f} GB free")

if gpu_mem_gb < 30:
    print(
        f"\nWARNING: {gpu_mem_gb:.0f} GB VRAM may not fit 128K context.\n"
        f"Consider A100, or reduce max_length in the config."
    )

## 2. Clone repo, install deps, HF login

In [ ]:
# ── EDIT THIS ──
REPO_URL = "https://github.com/YuvalShemla/LoSeM-attention.git"
BRANCH = "main"

REPO_DIR = "/content/LoSeM-attention"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!pip install -q transformers accelerate datasets pyyaml tqdm

from huggingface_hub import login
login()

## 3. Setup paths and config

In [ ]:
import yaml, json, time, subprocess, threading
from pathlib import Path

# Local SSD paths (ephemeral — we zip and download before runtime dies)
DATA_ROOT = Path("/content/extraction_data")
ZIP_DIR = Path("/content/zips")
DATA_ROOT.mkdir(exist_ok=True)
ZIP_DIR.mkdir(exist_ok=True)

# Load config
CONFIG_PATH = Path(REPO_DIR) / "src" / "extraction" / "extraction_config.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

p2_heads = config["phase2"].get("heads", "from_stats")
if p2_heads == "from_stats":
    p2_desc = (f"from_stats, top "
               f"{config['phase2']['n_heads_to_select']}")
elif p2_heads == "all":
    p2_desc = "all heads"
else:
    p2_desc = f"{len(p2_heads)} explicit pairs"

print("Config:")
print(f"  Model:      {config['model']['hf_name']}")
print(f"  Tasks:      {config['tasks']}")
print(f"  Layers:     {config['extraction']['layers']}")
print(f"  Max length: {config['extraction']['max_length']:,}")
print(f"  Phase 1:    {config['phase1']['examples_per_task']} ex/task, all heads")
print(f"  Phase 2:    {config['phase2']['examples_per_task']} ex/task, "
      f"heads: {p2_desc}")

In [ ]:
# ── Optional: override config ──
# Uncomment lines to change, then run this cell.

OVERRIDES = {
    # "extraction": {"max_length": 32768},
    # "extraction": {"layers": [0, 31]},
    # "tasks": ["math_calc", "passkey"],
    # "phase2": {"examples_per_task": 5},
}

if OVERRIDES:
    for section, values in OVERRIDES.items():
        if section in config and isinstance(config[section], dict):
            config[section].update(values)
        else:
            config[section] = values
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print("Config updated.")
else:
    print("No overrides.")

## 4. Background zip helper

After each task finishes extraction, we zip it in a background thread so extraction
of the next task can start immediately. Zips go to `/content/zips/`.

In [ ]:
zip_threads = []

def zip_in_background(src_dir, zip_path, delete_after=False):
    """Zip a directory in a background thread."""
    def _do_zip():
        t0 = time.time()
        src = Path(src_dir)
        subprocess.run(
            ["zip", "-r", "-q", str(zip_path), src.name],
            cwd=str(src.parent), check=True,
        )
        size_mb = Path(zip_path).stat().st_size / 1e6
        print(f"  [zip] {zip_path.name} ready ({size_mb:.0f} MB, {time.time()-t0:.0f}s)")
        if delete_after:
            shutil.rmtree(src_dir, ignore_errors=True)
            print(f"  [zip] deleted {src_dir} to free disk")

    t = threading.Thread(target=_do_zip, daemon=True)
    t.start()
    zip_threads.append(t)
    return t

def wait_for_zips():
    """Wait for all background zips to finish."""
    for t in zip_threads:
        t.join()
    print(f"All {len(zip_threads)} zips complete.")

def list_zips():
    """Show all ready zips with sizes."""
    zips = sorted(ZIP_DIR.glob("*.zip"))
    total = 0
    for z in zips:
        sz = z.stat().st_size
        total += sz
        print(f"  {z.name:40s}  {sz/1e6:8.0f} MB")
    print(f"  {'TOTAL':40s}  {total/1e6:8.0f} MB")
    return zips

## 5. Run Phase 1 — task by task

Extracts all heads for 1 example per task. After each task finishes:
- Zips that task's vectors in the background
- Starts extracting the next task immediately

Head statistics (tiny JSON files) are zipped once at the end.

In [ ]:
os.chdir(REPO_DIR)

from huggingface_hub import HfFolder
HF_TOKEN = HfFolder.get_token()
assert HF_TOKEN, "No HF token — re-run the login cell above"

tasks = config["tasks"]
out_cfg = config.get("output", {})
vectors_sub = out_cfg.get("vectors_subdir", "vectors/llama3.1_8b")

print(f"Phase 1: {len(tasks)} tasks, extracting one at a time\n")

for i, task in enumerate(tasks):
    print(f"{'='*50}")
    print(f"  [{i+1}/{len(tasks)}] {task}")
    print(f"{'='*50}")

    # Write a temp config with only this task
    task_config = dict(config)
    task_config["tasks"] = [task]
    tmp_cfg = f"/content/_cfg_{task}.yaml"
    with open(tmp_cfg, "w") as f:
        yaml.dump(task_config, f, default_flow_style=False, sort_keys=False)

    data_root = str(DATA_ROOT)
    token = HF_TOKEN

    # Run extraction
    !python -m src.extraction.extract_vectors \
        --phase 1 \
        --config {tmp_cfg} \
        --data-root {data_root} \
        --hf-token {token}

    os.remove(tmp_cfg) if os.path.exists(tmp_cfg) else None

    # Zip this task's vectors in background
    task_vec_dir = DATA_ROOT / vectors_sub / "all_heads" / task
    if task_vec_dir.exists():
        zip_path = ZIP_DIR / f"phase1_all_heads_{task}.zip"
        zip_in_background(task_vec_dir, zip_path, delete_after=True)
    else:
        print(f"  WARNING: {task_vec_dir} not found, skipping zip")

# Zip head statistics (small)
stats_dir = DATA_ROOT / out_cfg.get("head_stats_subdir", "head_statistics/llama3.1_8b")
if stats_dir.exists():
    zip_in_background(stats_dir, ZIP_DIR / "head_statistics.zip")

# Zip benchmarks (small)
bench_dir = DATA_ROOT / out_cfg.get("benchmarks_subdir", "benchmarks")
if bench_dir.exists():
    zip_in_background(bench_dir, ZIP_DIR / "benchmarks.zip")

wait_for_zips()
print("\nPhase 1 zips:")
list_zips();

## 6. Download Phase 1 zips

Run this cell to download all Phase 1 zips to your local machine.
Each zip is a separate browser download prompt.

In [ ]:
from google.colab import files

phase1_zips = sorted(ZIP_DIR.glob("phase1_*.zip")) + \
              sorted(ZIP_DIR.glob("head_statistics.zip")) + \
              sorted(ZIP_DIR.glob("benchmarks.zip"))

print(f"Downloading {len(phase1_zips)} files...\n")
for z in phase1_zips:
    sz_mb = z.stat().st_size / 1e6
    print(f"  {z.name} ({sz_mb:.0f} MB)")
    files.download(str(z))

## 7. Run Phase 2 — task by task

Same pattern: extract one task, zip in background, move to next.
Phase 2 needs head statistics from Phase 1 (still on disk).

**Note:** Phase 2 has 10 examples/task, so each task produces more data.
The `delete_after=True` flag frees disk after zipping so you don't run out of space.

In [ ]:
os.chdir(REPO_DIR)
zip_threads.clear()

print(f"Phase 2: {len(tasks)} tasks, extracting one at a time\n")

for i, task in enumerate(tasks):
    print(f"{'='*50}")
    print(f"  [{i+1}/{len(tasks)}] {task}")
    print(f"{'='*50}")

    task_config = dict(config)
    task_config["tasks"] = [task]
    tmp_cfg = f"/content/_cfg_{task}.yaml"
    with open(tmp_cfg, "w") as f:
        yaml.dump(task_config, f, default_flow_style=False, sort_keys=False)

    data_root = str(DATA_ROOT)
    token = HF_TOKEN

    !python -m src.extraction.extract_vectors \
        --phase 2 \
        --config {tmp_cfg} \
        --data-root {data_root} \
        --hf-token {token}

    os.remove(tmp_cfg) if os.path.exists(tmp_cfg) else None

    task_vec_dir = DATA_ROOT / vectors_sub / "selected_heads" / task
    if task_vec_dir.exists():
        zip_path = ZIP_DIR / f"phase2_selected_heads_{task}.zip"
        zip_in_background(task_vec_dir, zip_path, delete_after=True)
    else:
        print(f"  WARNING: {task_vec_dir} not found")

wait_for_zips()
print("\nPhase 2 zips:")
list_zips();

## 8. Download Phase 2 zips

In [ ]:
phase2_zips = sorted(ZIP_DIR.glob("phase2_*.zip"))

print(f"Downloading {len(phase2_zips)} files...\n")
for z in phase2_zips:
    sz_mb = z.stat().st_size / 1e6
    print(f"  {z.name} ({sz_mb:.0f} MB)")
    files.download(str(z))

## 9. Reassemble locally

After downloading all zips, reassemble the directory structure on your machine:

```bash
# Create the target directory
mkdir -p data/vectors/llama3.1_8b/all_heads
mkdir -p data/vectors/llama3.1_8b/selected_heads
mkdir -p data/head_statistics
mkdir -p data/benchmarks

# Unzip Phase 1 (one zip per task)
for f in phase1_all_heads_*.zip; do
    unzip -q "$f" -d data/vectors/llama3.1_8b/all_heads/
done

# Unzip Phase 2
for f in phase2_selected_heads_*.zip; do
    unzip -q "$f" -d data/vectors/llama3.1_8b/selected_heads/
done

# Unzip head stats and benchmarks
unzip -q head_statistics.zip -d data/
unzip -q benchmarks.zip -d data/
```

Final structure:
```
data/
├── vectors/llama3.1_8b/
│   ├── all_heads/{task}/ex_000/layer_XX.pt
│   └── selected_heads/{task}/ex_000/layer_XX.pt
├── head_statistics/llama3.1_8b/{task}.json
└── benchmarks/{task}.json
```

## Notes

- **Disk management:** Each task's unzipped vectors are deleted after zipping (`delete_after=True`).
  This means the local SSD only needs space for ~1 task of unzipped data + all zips at once.
  For Phase 1 that's roughly 8 GB/task unzipped; for Phase 2, ~13 GB/task.

- **Download timing:** The download cells use `files.download()` which triggers individual
  browser save prompts. You can also manually download from the Colab file browser
  (click the folder icon on the left → navigate to `/content/zips/`).

- **Colab timeouts:** Free Colab disconnects after ~90 min idle, Pro after ~24h.
  Keep the tab active. If the runtime dies mid-extraction, re-run from the task
  that was interrupted (edit `tasks` in the config override cell to skip completed tasks).

- **Resuming after disconnect:** Already-downloaded zips are safe on your machine.
  Only the current task's unzipped data is lost. Re-run with only the remaining tasks.

- **Smaller test runs:** Override `phase1.examples_per_task: 1`, `phase2.examples_per_task: 2`,
  and use 1-2 tasks to verify the pipeline quickly before doing the full extraction.